# Retrieval Augmented Generation (RAG) based chatbot

In [1]:
! git clone https://github.com/AshishJangra27/datasets

Cloning into 'datasets'...
remote: Enumerating objects: 367, done.
remote: Counting objects: 100% (132/132), done.
remote: Compressing objects: 100% (119/119), done.
remote: Total 367 (delta 30), reused 54 (delta 9), pack-reused 235 (from 1)
Receiving objects: 100% (367/367), 287.89 MiB | 12.94 MiB/s, done.
Resolving deltas: 100% (156/156), done.
Updating files: 100% (235/235), done.


In [2]:
import pandas as pd

df = pd.read_csv('/content/datasets/GFG Articles/data/0.csv')
df.head()

,id,title,author_name,author_id,tags,no_of_imgs,file_path,link,img_links,article
0,1047287.0,Refo Interview Experience for Backend Engineer,NaN,NaN,"Refo,Write it Up,Experiences,Interview Experie...",0.0,articles/1047287.txt,https://www.geeksforgeeks.org/refo-interview-e...,NaN,I got this opportunity through a referral The ...
1,1046950.0,KPIT Technologies Interview Experience for Ass...,NaN,NaN,"KPIT,On-Campus,Write it Up,Experiences,Intervi...",0.0,articles/1046950.txt,https://www.geeksforgeeks.org/kpit-technologie...,NaN,KPIT Technologies came to our campus with an A...
2,1046571.0,Bright data – An Industry Leading Web Data Pla...,rahulkwh,rahulkwh,"Spotlight,GBlog",1.0,articles/1046571.txt,https://www.geeksforgeeks.org/bright-data-an-i...,https://media.geeksforgeeks.org/wp-content/upl...,Web data collection is the process of gatherin...
3,1046311.0,Adobe Interview Experience,NaN,NaN,"Adobe,Write it Up,Experiences,Interview Experi...",0.0,articles/1046311.txt,https://www.geeksforgeeks.org/adobe-interview-...,NaN,Round 1It was a technical interview I was a bi...
4,1046286.0,Netaji Subhash Engineering College (NSEC) Camp...,NaN,NaN,"Career-Advices,NSEC,Write it Up,Campus Experie...",0.0,articles/1046286.txt,https://www.geeksforgeeks.org/netaji-subhash-e...,NaN,I am currently a 4thyear student of NSEC pursu...


In [3]:
len(df)

1700

In [4]:
from sentence_transformers import SentenceTransformer

# Load the model
model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

In [5]:
context = "Hello, how are you?"
embedding = model.encode(context)

print(embedding)

[ 0.00056076 -0.00376892 -0.00376892 ... -0.00454712 -0.00805664
  0.00274658]


In [6]:
len(embedding)

1024

In [16]:
from tqdm.auto import tqdm
embeddings = []
titles = []
articles = []

sample_df = df[:100]

for title, article in tqdm(zip(sample_df['title'], sample_df['article'])):
  titles.append(title)
  articles.append(article)
  embeddings.append(model.encode(article))

0it [00:00, ?it/s]

In [18]:
data = pd.DataFrame()

data['titles'] = titles
data['articles'] = articles
data['embeddings'] = embeddings

data.head()

,titles,articles,embeddings
0,Refo Interview Experience for Backend Engineer,I got this opportunity through a referral The ...,"[0.024169922, -0.03955078, -0.0095825195, -0.0..."
1,KPIT Technologies Interview Experience for Ass...,KPIT Technologies came to our campus with an A...,"[0.041992188, -0.09423828, -0.011779785, -0.05..."
2,Bright data – An Industry Leading Web Data Pla...,Web data collection is the process of gatherin...,"[-0.028198242, -0.027954102, -0.009033203, -0...."
3,Adobe Interview Experience,Round 1It was a technical interview I was a bi...,"[0.02331543, -0.03881836, -0.010009766, -0.001..."
4,Netaji Subhash Engineering College (NSEC) Camp...,I am currently a 4thyear student of NSEC pursu...,"[-0.0046691895, -0.09277344, -0.008666992, 0.0..."


In [21]:
query = 'adobe interview process'
query_embedding = model.encode(question)
len(query_embedding)

1024

In [40]:
import numpy as np

article_embeddings_np = np.array(list(data['embeddings']))
print(article_embeddings_np.shape)

similarity = model.similarity(query_embedding, article_embeddings_np)
print(similarity.shape)

data['similarity'] = similarity[0]
# data.head()

similarity_desc = data.sort_values(by='similarity', ascending=False)
similarity_desc.head()


(100, 1024)
torch.Size([1, 100])


,titles,articles,embeddings,similarity
3,Adobe Interview Experience,Round 1It was a technical interview I was a bi...,"[0.02331543, -0.03881836, -0.010009766, -0.001...",0.560571
19,Accenture Interview Experience (On-Campus),I am sharing my interview experience of Accent...,"[0.049316406, -0.076171875, -0.006591797, 0.00...",0.545989
69,Wipro Interview Experience for Project Engineer,My first interview was a nervewracking experie...,"[0.005065918, -0.048339844, -0.011108398, -0.0...",0.543388
59,Coditas Solution LLP Pune Interview Experience...,The company came for a campus placement in my ...,"[0.0032653809, -0.106933594, -0.00970459, -0.0...",0.504487
73,Google Interview Experience,I recently had the opportunity to interview wi...,"[-0.02319336, -0.05419922, -0.0045166016, -0.0...",0.497104


### Get top 5 relevant articles

In [41]:
def get_top_5(query):
  query_embedding = model.encode(query)
  similarity = model.similarity(query_embedding, article_embeddings_np)
  data['similarity'] = similarity[0]
  similarity_desc = data.sort_values(by='similarity', ascending=False)
  return similarity_desc[:5]

In [42]:
top_5 = get_top_5("What is time complexity")
top_5

,titles,articles,embeddings,similarity
53,ChatGPT Tutorial: ChatGPT-3.5 Guide for Beginners,Hi How are youCan you provide me with a soluti...,"[0.016845703, 0.044921875, -0.011962891, -0.08...",0.542726
94,C++ STL Cheat Sheet,The C STL Cheat Sheet provides short and conci...,"[0.023803711, -0.041503906, -0.013977051, -0.0...",0.492025
91,What is DSA | DSA Full Form,What is DSADSA is defined as a combination of ...,"[-0.032714844, -0.055664062, -0.0079956055, -0...",0.459791
31,Tech Mahindra Interview Experience (On-Campus),Tech Mahindra conducted a fiveround recruitmen...,"[0.00970459, -0.04296875, -0.015258789, -0.067...",0.446888
68,Basic String Operations with Implementation,In this post we will look into some of the bas...,"[0.0043945312, -0.04345703, -0.008300781, -0.0...",0.418832


In [45]:
for a in top_5['articles']:
  print(a)
  print('=============')

Hi How are youCan you provide me with a solution to this problemWrite the code for the Fibonacci sequenceImagine having a virtual buddy who can understand what youre saying and respond in a way that makes sense Thats what Chat GPT does It has been trained on a lot of information to make its responses sound natural and friendly almost like chatting with a real personIn this ChatGPT tutorial youll learn the basics of how to use ChatGPT35 from setting up your account to writing effective promptsThis tutorial is for anyone who wants to learn how to use ChatGPT35 Whether you are a beginner or an experienced user you will find this tutorial to be informative and helpfulBy the end of this tutorial you will be able to use ChatGPT35 to create a variety of content and get the most out of this powerful AI language modelWhat is ChatGPTChatGPT is a conversational AI ChatBot which gives you relevant solutions to your realtime problem whether its from your coding or cooking it gives you frequent solu

### Send the Question + Top 5 Relevant articles to query the LLM.

In [64]:
from google.colab import userdata
from google import genai
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

def ask_gemini(question):
  top_5_df = get_top_5(question)

  full_articles = []
  for index, row in top_5_df.iterrows():
    full_articles.append(f"Title: {row['titles']}\nContent: {row['articles']}")

  context = "\n\n---\n\n".join(full_articles)

  prompt = f"""You are a chatbot who answers **ONLY** based on the context provided to you.
  If the context does not contain the answer to my question, respond "I don't have enough data to answer your question".
  **NEVER** invent or use information outside of the context provided.
  Respond in less than 200 words.

  QUESTION: {question}
  CONTEXT:
  {context}"""

  client = genai.Client(api_key=GEMINI_API_KEY)

  response = client.models.generate_content(
      model="gemini-2.5-flash",
      contents=prompt
  )

  print(response.text)

In [69]:
ask_gemini('Who discovered Pluto')

I don't have enough data to answer your question.


In [62]:
ask_gemini('What is time complexity')

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The request timed out. Please try again.', 'status': 'UNAVAILABLE'}}

In [70]:
aks_gemini('How many rounds does Adobe interview process have?')

Based on the provided context for "Adobe Interview Experience", the interview process has **4 rounds**:

1.  Round 1 (Technical Interview)
2.  Round 2 (Discussion about projects, simple DP question)
3.  Round 3 (Short interview, problem-solving)
4.  HR Interview


In [67]:
ask_gemini('What is the most significant cereal in the world')

Wheat is the most popular and significant cereal in the world for both human and animal nutrition.


### Conclusion

1. High Quality with Grounded Source Context: When asking domain-specific questions (like How many rounds does Adobe interview process have?), the system retrieves relevant documents and successfully synthesizes an accurate, factual response using Gemini. Verified with the actual article.
2. Effective Safeguards Against Hallucination: When asking off-topic questions (like Who discovered Pluto), the system correctly replies with "I don't have enough data to answer your question" because it relies strictly on the provided context.
3. Context Window vs. API Timeouts: Sending the full length of 5 dense articles verbatim to the API can trigger server-side/network gateway timeouts (503 ServerError). Perform document chunking to resolve the timeout issue entirely, while still keeping enough localized information for the model to answer accurately.
4. Vector Database for Scale: Embeddings are currently kept in a pandas DataFrame in memory, and every query does a brute-force similarity scan against all of them. This is fine for 100 articles, but it won't scale. The embeddings are also lost when the session ends, so they have to be re-computed every run. A vector database (e.g. ChromaDB, FAISS, Pinecone, pgvector) fixes this by persisting embeddings, using approximate nearest-neighbour (ANN) indexes for fast search over millions of vectors, and supporting metadata filtering (e.g. by title or date) alongside similarity search.